In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
!pip install flask pyngrok

In [9]:
!ngrok config add-authtoken 2yzBts6AwWwzG0NL7rCxccWziQk_5oUBwnPjwjiKhSe7TpDij

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [10]:
import torch
from torchvision import models, transforms
from flask import Flask, request, jsonify
from PIL import Image
import scipy.io
from pyngrok import ngrok
import io

# === モデルとクラス名の準備 ===
meta = scipy.io.loadmat('/content/drive/MyDrive/car_devkit/devkit/cars_meta.mat')
class_names = [c[0] for c in meta['class_names'][0]]  # 0-indexed 196クラス

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.efficientnet_b2(weights=None)
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, 196)
model.load_state_dict(torch.load('/content/drive/MyDrive/efficientnetb2_car_model.pth', map_location=device))
model = model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# === Flask アプリの作成 ===
app = Flask(__name__)

@app.route('/')
def home():
    return '🚗 Stanford Cars Classifier is Running! Use /predict'

@app.route('/predict', methods=['POST'])
def predict():
    if 'file' not in request.files:
        return jsonify({'error': 'ファイルがありません'}), 400

    file = request.files['file']
    image = Image.open(file.stream).convert('RGB')
    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        pred_id = torch.argmax(output, dim=1).item()
        pred_class = class_names[pred_id]

    return jsonify({
        'class_id': pred_id,
        'class_name': pred_class
    })

# === ngrokトンネルを張ってFlaskを外部公開 ===
port = 5000
public_url = ngrok.connect(port)
print(f"🌐 Public URL: {public_url}")

app.run(port=port)


🌐 Public URL: NgrokTunnel: "https://d02c-35-243-146-106.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [25/Jun/2025 05:07:08] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Jun/2025 05:07:08] "GET /favicon.ico HTTP/1.1" 404 -


In [11]:
import requests

url = 'https://xxxxxx.ngrok-free.app/predict'  # ← 公開URLに置き換えて
files = {'file': open('/content/drive/MyDrive/2013-ford-expedition-king-ranch.jpg', 'rb')}
response = requests.post(url, files=files)

print(response.json())


JSONDecodeError: Expecting value: line 1 column 1 (char 0)